# Moduł 3: NumPy — Wektory, macierze i losowość

NumPy to **fundament** obliczeń naukowych w Pythonie. Tensory w PyTorchu działają niemal identycznie.

W tym module formalizujemy kluczowe pojęcia algebry liniowej i rachunku prawdopodobieństwa, które stanowią matematyczną bazę uczenia maszynowego i głębokiego uczenia.

## Spis treści
1. Tworzenie tablic (tensory)
2. Indeksowanie i wycinanie
3. Operacje na tablicach
4. Algebra liniowa (wektory i macierze)
5. Normy wektorów i macierzy
6. Wartości własne i rozkład macierzy
7. Losowość — rozkłady prawdopodobieństwa
8. Broadcasting
9. **Zadania do samodzielnego rozwiązania**

---

In [ ]:
import numpy as np

## 1. Tworzenie tablic (tensory)

Tablica NumPy (`ndarray`) to **wielowymiarowa siatka liczb tego samego typu**. W odróżnieniu od listy Pythona, operacje na tablicach są **wektoryzowane** (szybkie, bez pętli).

### Tensor — formalna definicja

Tensor to uogólnienie skalarów, wektorów i macierzy na dowolną liczbę wymiarów:

| Ranga | Nazwa | Kształt | Przykład w ML |
|-------|-------|---------|---------------|
| 0 | Skalar | $()$ | wartość loss |
| 1 | Wektor | $(n,)$ | embedding słowa |
| 2 | Macierz | $(m, n)$ | batch embeddingów |
| 3 | Tensor 3D | $(b, m, n)$ | batch sekwencji |
| 4 | Tensor 4D | $(b, c, h, w)$ | batch obrazów (NCHW) |

Formalnie: tensor rzędu $k$ to element $\mathcal{T} \in \mathbb{R}^{d_1 \times d_2 \times \cdots \times d_k}$.

In [ ]:
# Z listy Pythona
a = np.array([1, 2, 3, 4, 5])
print(a) # [1 2 3 4 5]
print(a.shape) # (5,) — wektor 5-elementowy
print(a.dtype) # int64

# Macierz 2D (lista list)
M = np.array([[1, 2, 3],
 [4, 5, 6]])
print(M.shape) # (2, 3) — 2 wiersze, 3 kolumny

# Specjalne tablice
print(np.zeros((3, 4))) # macierz 3x4 samych zer
print(np.ones((2, 2))) # macierz 2x2 samych jedynek
print(np.eye(3)) # macierz jednostkowa 3x3
print(np.arange(0, 10, 2)) # [0 2 4 6 8] — jak range, ale numpy
print(np.linspace(0, 1, 5)) # [0. 0.25 0.5 0.75 1. ] — 5 równo rozłożonych

## 2. Indeksowanie i wycinanie (slicing)

Działa jak w listach Pythona, ale rozszerzone na wiele wymiarów. Kluczowa różnica: **slice zwraca view** (widok na te same dane), a nie kopię!

> **View vs. Copy**: `a[1:3]` to view (zmiana widoku zmienia oryginał). Aby uzyskać kopię: `a[1:3].copy()`.

**Boolean indexing** (`a[a > 5]`) — niezwykle ważny w ML do filtrowania danych i tworzenia masek.

In [ ]:
M = np.array([[10, 20, 30],
 [40, 50, 60],
 [70, 80, 90]])

print(M[0, 0]) # 10 — element (wiersz 0, kolumna 0)
print(M[1, 2]) # 60 — element (wiersz 1, kolumna 2)
print(M[0]) # [10 20 30] — cały pierwszy wiersz
print(M[:, 1]) # [20 50 80] — cała druga kolumna
print(M[0:2, 1:3]) # [[20 30] [50 60]] — podmacierz

# Indeksowanie boolowskie (maski)
print(M > 50) # macierz True/False
print(M[M > 50]) # [60 70 80 90] — elementy > 50
M[M > 50] = 0 # zamień elementy > 50 na 0
print(M)

## 3. Operacje na tablicach

NumPy wykonuje operacje **element-wise** (po elemencie) automatycznie — bez pisania pętli. To tzw. **wektoryzacja**, która jest 10-100x szybsza niż pętle Pythona (dzięki implementacji w C).

### Agregacje z parametrem `axis`

- `axis=0`: agregacja **wzdłuż wierszy** (wynik: jeden wiersz)
- `axis=1`: agregacja **wzdłuż kolumn** (wynik: jedna kolumna)
- `axis=None` (domyślnie): agregacja **całej tablicy** (wynik: skalar)

Formalnie, dla macierzy $A \in \mathbb{R}^{m \times n}$:
$$\text{sum}(A, \text{axis}=0)_j = \sum_{i=1}^{m} A_{ij} \in \mathbb{R}^n, \quad \text{sum}(A, \text{axis}=1)_i = \sum_{j=1}^{n} A_{ij} \in \mathbb{R}^m$$

In [ ]:
a = np.array([1, 2, 3, 4])
b = np.array([10, 20, 30, 40])

# Operacje elementowe
print(a + b) # [11 22 33 44]
print(a * b) # [10 40 90 160] — NIE mnożenie macierzowe!
print(a ** 2) # [1 4 9 16]
print(np.sqrt(a)) # [1. 1.41 1.73 2. ]

# Funkcje agregujące
print(a.sum()) # 10
print(a.mean()) # 2.5
print(a.std()) # odchylenie standardowe
print(a.min(), a.max()) # 1, 4
print(a.argmax()) # 3 — indeks największego elementu

# Agregacja po osiach (dla macierzy)
M = np.array([[1, 2, 3],
 [4, 5, 6]])
print(M.sum(axis=0)) # [5 7 9] — suma po kolumnach (w dół)
print(M.sum(axis=1)) # [6 15] — suma po wierszach (w prawo)

## 4. Algebra liniowa — wektory i macierze

Algebra liniowa to **język uczenia maszynowego**. Prawie każdy model ML można wyrazić jako operacje na wektorach i macierzach.

### Iloczyn skalarny (dot product)

$$\mathbf{a} \cdot \mathbf{b} = \sum_{i=1}^{n} a_i b_i = \|\mathbf{a}\|_2 \cdot \|\mathbf{b}\|_2 \cdot \cos\theta$$

Interpretacja geometryczna: rzut $\mathbf{a}$ na $\mathbf{b}$ pomnożony przez $\|\mathbf{b}\|$. W ML: miara podobieństwa wektorów (attention, cosine similarity).

### Mnożenie macierzy

Dla $A \in \mathbb{R}^{m \times n}$ i $B \in \mathbb{R}^{n \times p}$:

$$(AB)_{ij} = \sum_{k=1}^{n} A_{ik} B_{kj}, \quad AB \in \mathbb{R}^{m \times p}$$

Złożoność: $O(mnp)$. W sieciach neuronowych: $\mathbf{y} = W\mathbf{x} + \mathbf{b}$ (warstwa liniowa).

### Transpozycja

$$[A^T]_{ij} = A_{ji}$$

Własności: $(AB)^T = B^T A^T$, $(A^T)^T = A$, $(A + B)^T = A^T + B^T$.

### Macierz odwrotna

$$A A^{-1} = A^{-1} A = I$$

Istnieje tylko dla macierzy kwadratowych z $\det(A) \neq 0$. W ML: Normal Equation $\hat{\mathbf{w}} = (X^TX)^{-1}X^T\mathbf{y}$.

### Ślad macierzy (trace)

$$\text{tr}(A) = \sum_{i=1}^{n} A_{ii}$$

Własności: $\text{tr}(A + B) = \text{tr}(A) + \text{tr}(B)$, $\text{tr}(AB) = \text{tr}(BA)$, $\text{tr}(A) = \sum_i \lambda_i$.

### Wyznacznik (determinant)

$$\det(A)$$ — miara, o ile macierz „rozciąga" przestrzeń. $|\det(A)|$ = stosunek objętości po transformacji.

- $\det(A) = 0 \implies$ macierz **osobliwa** (brak odwrotnej, kolumny liniowo zależne)
- $\det(AB) = \det(A) \cdot \det(B)$

### Macierze ortogonalne

$Q$ jest ortogonalna, jeśli $Q^T Q = Q Q^T = I$ (tj. $Q^{-1} = Q^T$).

Kolumny macierzy ortogonalnej tworzą **ortonormalną bazę**. W PCA wektory własne macierzy kowariancji tworzą bazę ortogonalną.

In [ ]:
# Iloczyn skalarny wektorów
a = np.array([1, 2, 3])
b = np.array([4, 5, 6])
print(np.dot(a, b)) # 1*4 + 2*5 + 3*6 = 32
print(a @ b) # to samo — operator @

# Norma (długość) wektora: ||v|| = sqrt(v·v)
print(np.linalg.norm(a)) # sqrt(1+4+9) = sqrt(14) ≈ 3.74

# Mnożenie macierzy
A = np.array([[1, 2], # 2x2
 [3, 4]])
B = np.array([[5, 6], # 2x2
 [7, 8]])

C = A @ B # mnożenie macierzowe
print(C) # [[19 22] [43 50]]

# UWAGA: A * B to mnożenie elementowe, A @ B to macierzowe!
print(A * B) # [[5 12] [21 32]] — element po elemencie

In [ ]:
# Transpozycja — zamiana wierszy z kolumnami
M = np.array([[1, 2, 3],
 [4, 5, 6]])
print(M.T) # [[1, 4], [2, 5], [3, 6]]
print(M.shape) # (2, 3)
print(M.T.shape) # (3, 2)

# Reshape — zmiana kształtu bez zmiany danych
v = np.arange(12) # [0, 1, 2, ..., 11]
print(v.reshape(3, 4)) # macierz 3x4
print(v.reshape(2, 2, 3)) # tensor 3D: 2x2x3

# Flatten — spłaszczenie do 1D (użyteczne w sieciach neuronowych!)
M_flat = M.flatten()
print(M_flat) # [1 2 3 4 5 6]

## 5. Normy wektorów i macierzy

**Norma** mierzy „wielkość" wektora lub macierzy. Jest kluczowa w regularyzacji (L1, L2), funkcjach straty i optymalizacji.

### Normy wektorowe ($L^p$-normy)

Ogólna $L^p$-norma wektora $\mathbf{x} \in \mathbb{R}^n$:

$$\|\mathbf{x}\|_p = \left(\sum_{i=1}^{n} |x_i|^p\right)^{1/p}$$

| Norma | Wzór | NumPy | Zastosowanie w ML |
|-------|------|-------|-------------------|
| $L^1$ (Manhattan) | $\|\mathbf{x}\|_1 = \sum_i |x_i|$ | `np.linalg.norm(x, 1)` | Regularyzacja Lasso (sparse weights) |
| $L^2$ (Euklidesowa) | $\|\mathbf{x}\|_2 = \sqrt{\sum_i x_i^2}$ | `np.linalg.norm(x)` | Regularyzacja Ridge, odległości |
| $L^\infty$ (Max) | $\|\mathbf{x}\|_\infty = \max_i |x_i|$ | `np.linalg.norm(x, np.inf)` | Ograniczenia adwersarialne |

### Normy macierzowe

**Norma Frobeniusa** — uogólnienie $L^2$ na macierze:

$$\|A\|_F = \sqrt{\sum_{i,j} A_{ij}^2} = \sqrt{\text{tr}(A^T A)}$$

### Cosine similarity

Miara kątowego podobieństwa wektorów (niezależna od ich długości):

$$\cos(\mathbf{a}, \mathbf{b}) = \frac{\mathbf{a} \cdot \mathbf{b}}{\|\mathbf{a}\|_2 \cdot \|\mathbf{b}\|_2} \in [-1, 1]$$

Używana w embeddingach słów (Word2Vec, GloVe) i mechanizmie attention.

### Liczba uwarunkowania (condition number)

$$\kappa(A) = \|A\| \cdot \|A^{-1}\| = \frac{\sigma_{\max}}{\sigma_{\min}}$$

- $\kappa(A) \approx 1$ → dobrze uwarunkowana (stabilna numerycznie)
- $\kappa(A) \gg 1$ → źle uwarunkowana (małe zmiany wejścia → duże zmiany wyniku)

Źle uwarunkowane macierze pojawiają się np. przy **kolinearnych cechach** — dlatego stosujemy regularyzację.


In [ ]:
import numpy as np

x = np.array([3, -4, 0, 2])

# Normy wektorowe
print("L1 (Manhattan):", np.linalg.norm(x, 1))      # sum(|xi|) = 9
print("L2 (Euklidesowa):", np.linalg.norm(x, 2))     # sqrt(sum(xi^2))
print("L-inf (Max):", np.linalg.norm(x, np.inf))      # max(|xi|) = 4

# Norma Frobeniusa macierzy
A = np.array([[1, 2], [3, 4]])
print("\nFrobenius norm:", np.linalg.norm(A, 'fro'))

# Cosine similarity
a = np.array([1, 2, 3])
b = np.array([4, 5, 6])
cos_sim = np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))
print("\nCosine similarity:", cos_sim)

# Condition number
print("Condition number A:", np.linalg.cond(A))


## 6. Wartości własne i rozkład macierzy

### Definicja wartości własnych

Wektor $\mathbf{v} \neq \mathbf{0}$ jest **wektorem własnym** macierzy $A \in \mathbb{R}^{n \times n}$ z **wartością własną** $\lambda$, jeśli:

$$A\mathbf{v} = \lambda \mathbf{v}$$

Równoważnie: $\det(A - \lambda I) = 0$ (wielomian charakterystyczny stopnia $n$).

### Dekompozycja spektralna (eigendecomposition)

Jeśli $A$ ma $n$ liniowo niezależnych wektorów własnych:

$$A = Q \Lambda Q^{-1}$$

gdzie $Q = [\mathbf{v}_1 | \cdots | \mathbf{v}_n]$ (macierz wektorów własnych), $\Lambda = \text{diag}(\lambda_1, \ldots, \lambda_n)$.

Dla macierzy **symetrycznej** ($A = A^T$): $Q$ jest ortogonalna ($Q^{-1} = Q^T$), więc $A = Q\Lambda Q^T$.

### Singular Value Decomposition (SVD)

Każdą macierz $A \in \mathbb{R}^{m \times n}$ można rozłożyć:

$$A = U \Sigma V^T$$

- $U \in \mathbb{R}^{m \times m}$ — lewe wektory singularne (ortogonalna)
- $\Sigma \in \mathbb{R}^{m \times n}$ — wartości singularne $\sigma_1 \geq \sigma_2 \geq \cdots \geq 0$ na diagonali
- $V \in \mathbb{R}^{n \times n}$ — prawe wektory singularne (ortogonalna)

**Truncated SVD** zachowuje $k$ największych wartości singularnych → **redukcja wymiarowości** (jak PCA).

### Związek z PCA

PCA to eigendecomposition macierzy kowariancji:

$$C = \frac{1}{n-1} X^T X, \quad C\mathbf{v}_i = \lambda_i \mathbf{v}_i$$

- Wektory własne $\mathbf{v}_i$ → **kierunki głównych składowych** (principal components)
- Wartości własne $\lambda_i$ → **wariancja wyjaśniona** przez każdą składową
- **Explained Variance Ratio**: $\text{EVR}_i = \frac{\lambda_i}{\sum_j \lambda_j}$


In [ ]:
# Wartości własne i wektory własne
A = np.array([[4, 2], [1, 3]])
eigenvalues, eigenvectors = np.linalg.eig(A)
print("Wartości własne:", eigenvalues)
print("Wektory własne (kolumny):\n", eigenvectors)

# Weryfikacja: A @ v = lambda * v
for i in range(len(eigenvalues)):
    v = eigenvectors[:, i]
    print(f"\nA @ v{i} = {A @ v}")
    print(f"lambda{i} * v{i} = {eigenvalues[i] * v}")

# SVD
U, sigma, Vt = np.linalg.svd(A)
print("\nSVD — wartości singularne:", sigma)

# Macierz symetryczna (kowariancji) — eigendecomposition = PCA
np.random.seed(42)
X = np.random.randn(100, 3)
X -= X.mean(axis=0)
cov = (X.T @ X) / (len(X) - 1)
evals, evecs = np.linalg.eigh(cov)  # eigh dla macierzy symetrycznych
evr = evals[::-1] / evals.sum()
print("\nExplained Variance Ratio:", evr)


## 7. Losowość i rozkłady prawdopodobieństwa

Losowość w ML: inicjalizacja wag, tasowanie danych, augmentacja, dropout, stochastic gradient descent.

**Ziarno losowości (seed)** zapewnia **powtarzalność**: ten sam seed → te same „losowe" liczby (generator pseudolosowy).

### Kluczowe rozkłady prawdopodobieństwa

#### Rozkład jednostajny (Uniform)

$$f(x) = \frac{1}{b - a}, \quad x \in [a, b]$$

$$\mathbb{E}[X] = \frac{a+b}{2}, \quad \text{Var}(X) = \frac{(b-a)^2}{12}$$

Używany w: inicjalizacji wag (Xavier uniform), losowym przeszukiwaniu hiperparametrów.

#### Rozkład normalny (Gaussian)

$$f(x) = \frac{1}{\sigma\sqrt{2\pi}} \exp\left(-\frac{(x - \mu)^2}{2\sigma^2}\right)$$

$$\mathbb{E}[X] = \mu, \quad \text{Var}(X) = \sigma^2$$

**Reguła 68-95-99.7**: odpowiednio $\pm 1\sigma$, $\pm 2\sigma$, $\pm 3\sigma$ obejmuje 68%, 95%, 99.7% danych.

Używany w: inicjalizacji wag (Kaiming, Xavier), Batch Normalization, modele generatywne (VAE, diffusion).

#### Rozkład Bernoulliego

$$P(X = k) = p^k (1-p)^{1-k}, \quad k \in \{0, 1\}$$

$$\mathbb{E}[X] = p, \quad \text{Var}(X) = p(1-p)$$

Używany w: klasyfikacja binarna, Dropout (maska Bernoulliego z $p = 1 - \text{drop\_rate}$).

#### Rozkład dwumianowy (Binomial)

$$P(X = k) = \binom{n}{k} p^k (1-p)^{n-k}$$

Uogólnienie Bernoulliego na $n$ niezależnych prób.

### Inicjalizacja wag

Dobra inicjalizacja zapobiega vanishing/exploding gradients:

| Metoda | Rozkład | Wariancja | Kiedy stosować |
|--------|---------|-----------|----------------|
| Xavier (Glorot) | $\mathcal{U}\left(-\sqrt{\frac{6}{n_{in}+n_{out}}}, \sqrt{\frac{6}{n_{in}+n_{out}}}\right)$ | $\frac{2}{n_{in}+n_{out}}$ | sigmoid, tanh |
| Kaiming (He) | $\mathcal{N}\left(0, \frac{2}{n_{in}}\right)$ | $\frac{2}{n_{in}}$ | ReLU, LeakyReLU |

In [ ]:
# Ustawianie ziarna — powtarzalność!
np.random.seed(42)

# Losowe liczby
print(np.random.rand(3)) # 3 losowe z [0, 1)
print(np.random.rand(2, 3)) # macierz 2x3 losowych z [0, 1)
print(np.random.randint(1, 7, size=10)) # 10 losowych int z [1, 7) — jak rzut kostką

# Rozkład normalny N(0, 1)
print(np.random.randn(5)) # 5 losowych z rozkładu normalnego

# Losowy wybór z tablicy
kolory = ["czerwony", "niebieski", "zielony"]
print(np.random.choice(kolory, size=5)) # losowe kolory

# Tasowanie (shuffle)
dane = np.arange(10)
np.random.shuffle(dane)
print(dane) # losowa kolejność

In [ ]:
# Demonstracja: ten sam seed = te same wyniki
np.random.seed(123)
print("Próba 1:", np.random.rand(3))

np.random.seed(123) # resetujemy seed
print("Próba 2:", np.random.rand(3)) # identyczne wyniki!

# Bez resetu — inne wyniki
print("Próba 3:", np.random.rand(3)) # inne

## 8. Broadcasting

Broadcasting pozwala NumPy automatycznie „rozciągać" mniejszą tablicę do wymiarów większej podczas operacji element-wise, **bez kopiowania danych** w pamięci.

### Formalne reguły kompatybilności

Dwie tablice o kształtach $(d_1, d_2, \ldots, d_k)$ i $(e_1, e_2, \ldots, e_m)$ są kompatybilne, jeśli:

1. **Dopełnienie wymiarów**: krótszy kształt jest uzupełniany jedynkami **od lewej** (np. $(3,) \to (1, 3)$)
2. **Kompatybilność osi**: po wyrównaniu, dla każdej osi wymiary muszą być **równe** lub jeden z nich wynosi **1**
3. **Rozciągnięcie**: wymiar o rozmiarze $1$ jest replikowany wzdłuż osi do rozmiaru drugiej tablicy

**Przykład**: $(2, 3) + (3,) \to (2, 3) + (1, 3) \to (2, 3)$ — wektor dodany do każdego wiersza.

**Uwaga**: Broadcasting jest kluczowy w ML — pozwala pisać zwektoryzowaną normalizację, obliczanie odległości i scoring bez jawnych pętli:

$$X_{\text{norm}} = \frac{X - \boldsymbol{\mu}}{\boldsymbol{\sigma}}$$

gdzie $\boldsymbol{\mu}, \boldsymbol{\sigma} \in \mathbb{R}^{1 \times d}$ są broadcastowane wzdłuż wymiaru próbek.

In [ ]:
# Skalar + tablica
a = np.array([1, 2, 3])
print(a + 10) # [11 12 13] — 10 jest "rozciągnięte"

# Wektor + macierz
M = np.array([[1, 2, 3],
 [4, 5, 6]])
v = np.array([10, 20, 30])
print(M + v) # [[11 22 33] [14 25 36]]
# v jest dodany do KAŻDEGO wiersza

# Kolumna + macierz
col = np.array([[100], [200]]) # shape (2, 1)
print(M + col) # [[101 102 103] [204 205 206]]
# col jest dodana do KAŻDEJ kolumny

# Praktyczny przykład: normalizacja (odjęcie średniej)
dane = np.array([[10, 20, 30],
 [40, 50, 60]])
srednie_kolumn = dane.mean(axis=0) # [25, 35, 45]
dane_znormalizowane = dane - srednie_kolumn
print(dane_znormalizowane) # [[-15 -15 -15] [15 15 15]]

---
---
## Zadania do samodzielnego rozwiązania

---

### Zadanie 1: Statystyki wektora (łatwe)

Wygeneruj 1000 losowych liczb z rozkładu normalnego N(5, 2) — średnia 5, odchylenie 2.
Oblicz i wypisz: średnią, medianę, odchylenie standardowe, min, max.

In [ ]:
# Zadanie 1
np.random.seed(42)
# TWÓJ KOD TUTAJ

### Zadanie 2: Mnożenie macierzy ręcznie (średnie)

Napisz funkcję `mnozenie_macierzy(A, B)`, która mnoży dwie macierze **bez użycia** `np.dot` ani `@`. Użyj pętli.

Sprawdź wynik porównując z `A @ B`.

In [ ]:
# Zadanie 2
def mnozenie_macierzy(A, B):
 # TWÓJ KOD TUTAJ
 pass

A = np.array([[1, 2], [3, 4]])
B = np.array([[5, 6], [7, 8]])
print("Ręczne:", mnozenie_macierzy(A, B))
print("NumPy: ", A @ B)

### Zadanie 3: Normalizacja Min-Max (średnie)

Napisz funkcję `min_max_normalizacja(X)`, która normalizuje macierz kolumna po kolumnie:

$$X_{\text{norm}} = \frac{X - X_{\min}}{X_{\max} - X_{\min}}$$

Po normalizacji każda kolumna powinna mieć wartości z zakresu [0, 1].

In [ ]:
# Zadanie 3
def min_max_normalizacja(X):
 # TWÓJ KOD TUTAJ (użyj broadcastingu!)
 pass

dane = np.array([[10, 200, 3000],
 [20, 400, 1000],
 [30, 100, 5000]])
print(min_max_normalizacja(dane))

### Zadanie 4: Cosine Similarity (średnie)

Napisz funkcję `cosine_similarity(a, b)`, która oblicza podobieństwo kosinusowe dwóch wektorów:

$$\text{sim}(\vec{a}, \vec{b}) = \frac{\vec{a} \cdot \vec{b}}{\|\vec{a}\| \cdot \|\vec{b}\|}$$

Wynik 1.0 = identyczne kierunki, 0 = prostopadłe, -1 = przeciwne.

In [ ]:
# Zadanie 4
def cosine_similarity(a, b):
 # TWÓJ KOD TUTAJ
 pass

v1 = np.array([1, 0, 0])
v2 = np.array([0, 1, 0])
v3 = np.array([1, 0, 0])
v4 = np.array([-1, 0, 0])

print(cosine_similarity(v1, v3)) # 1.0 (identyczne)
print(cosine_similarity(v1, v2)) # 0.0 (prostopadłe)
print(cosine_similarity(v1, v4)) # -1.0 (przeciwne)

### Zadanie 5: Symulacja Monte Carlo — przybliżanie π (trudne)

Wygeneruj N losowych punktów (x, y) z kwadratu [0, 1] × [0, 1].

Policz ile punktów wpada do ćwiartki koła (odległość od (0,0) ≤ 1).

Przybliżenie π: $\pi \approx 4 \cdot \frac{\text{punkty w kole}}{N}$

Oblicz π dla N = 100, 1000, 10000, 100000, 1000000.

In [ ]:
# Zadanie 5: Monte Carlo — przybliżanie π
np.random.seed(42)

for N in [100, 1000, 10_000, 100_000, 1_000_000]:
 # TWÓJ KOD TUTAJ
 # pi_przyblizenie = ...
 # print(f"N={N:>10,}: π ≈ {pi_przyblizenie:.6f}")
 pass

### Zadanie 6: Softmax (trudne)

Zaimplementuj funkcję softmax — kluczową w sieciach neuronowych (klasyfikacja).

$$\text{softmax}(x_i) = \frac{e^{x_i}}{\sum_{j} e^{x_j}}$$

Wynik to wektor prawdopodobieństw (wszystkie ≥ 0, suma = 1).

**Wskazówka stabilności:** Odejmij `max(x)` przed `exp`, aby uniknąć overflow.

In [ ]:
# Zadanie 6: Softmax
def softmax(x):
 # TWÓJ KOD TUTAJ
 pass

logits = np.array([2.0, 1.0, 0.1])
probs = softmax(logits)
print(probs) # [0.659, 0.242, 0.099] (w przybliżeniu)
print(probs.sum()) # 1.0

### Zadanie 7: Macierz kowariancji (srednie)

Wygeneruj 3 skorelowane zmienne losowe (1000 probek):
- $x_1 \sim N(0, 1)$
- $x_2 = 2x_1 + \text{szum}$ (silna korelacja z $x_1$)
- $x_3 \sim N(0, 1)$ (niezalezna)

1. Oblicz macierz kowariancji za pomoca `np.cov()`
2. Oblicz te sama macierz **recznie** (wzor: $\text{Cov}(X,Y) = \frac{1}{n-1}\sum(x_i - \bar{x})(y_i - \bar{y})$)
3. Porownaj wyniki — czy sa identyczne?

In [ ]:
# Zadanie 7: Macierz kowariancji
np.random.seed(42)
# TWOJ KOD TUTAJ

### Zadanie 8: Broadcasting puzzle (trudne)

Masz macierz A o ksztalcie (5, 3) i wektor b o ksztalcie (3,).

Bez uzycia petli ani `np.tile`:
1. Odejmij srednia kazdej kolumny od A (centrowanie)
2. Podziel kazdy wiersz przez norme euklidesowa tego wiersza (normalizacja wierszy)
3. Oblicz macierz odleglosci euklidesowych miedzy wszystkimi parami wierszy (wynik: macierz 5x5)

Wskazowka: Uzyj broadcastingu i `np.newaxis`.

In [ ]:
# Zadanie 8: Broadcasting puzzle
np.random.seed(42)
A = np.random.randn(5, 3)
print("A:\n", A)
# TWOJ KOD TUTAJ

### Zadanie 9: Normalizacja Min-Max per kolumna
Wygeneruj macierz 100x5 z losowymi wartościami z rozkładu normalnego.
Znormalizuj każdą kolumnę do zakresu [0, 1] używając wzoru:

$x_{norm} = \frac{x - x_{min}}{x_{max} - x_{min}}$

Zweryfikuj, że min i max każdej kolumny to odpowiednio 0 i 1.
Nie używaj pętli — tylko operacje NumPy z broadcasting.

In [ ]:
import numpy as np

# TWOJ KOD TUTAJ


### Zadanie 10: Macierz odległości euklidesowych
Mając zbiór N=50 punktów w 3D (macierz Nx3), oblicz macierz odległości NxN.
Element (i,j) to odległość euklidesowa między punktem i a j.

Zrób to **bez pętli** — użyj broadcasting i wektoryzacji NumPy.

**Wskazowka:** $\|a - b\|^2 = \|a\|^2 + \|b\|^2 - 2 a^T b$

In [ ]:
import numpy as np

# TWOJ KOD TUTAJ
